# Evaporative Cooling — Harmonic Oscillator Trap  (N, Ω) formulation

Semiclassical simulation using the **(N, Ω)** recurrence pair.

The density of states gives polylogarithm orders $s_N = 3$ (particle number)
and $s_\Omega = 4$ (grand potential).  Temperature exponent $p = 3$.

**Reference:** Arvizu-Velázquez *et al.* (2026); Poveda-Cuevas (2026).

In [ ]:
import numpy as np
import mpmath as mp
import time
from matplotlib import pyplot as plt

from evap_cool_utils_omega import (
    ConstantsEV,
    g_tilde, g_bar,
    newton_raphson_1var,
    mb_particle_number, mb_temperature_oscillator,
    create_omega_result_dict, create_mb_result_dict,
    build_cutoff_schedule, initialize_omega_state, initialize_mb_state,
    run_quantum_evaporation_omega, run_mb_evaporation,
    compute_initial_omega,
    make_truncated_NOmega, make_nr_solver_omega, make_eos_energy,
    plot_combined_overview, plot_individual_panels,
)

## 1. Physical Parameters

In [ ]:
h = ConstantsEV.h; hb = ConstantsEV.hbar; kB = ConstantsEV.kB; m = ConstantsEV.m_Na23
omega = 2 * np.pi * 100
N0 = 1e7; T0 = 5e-5

def dos_prefactor(T):
    return (kB * T)**3 / (hb * omega)**3

## 2. Equation of State

In [ ]:
def N_equation_boson(x):
    return dos_prefactor(T0) * mp.polylog(3, mp.exp(x)) - N0
def dN_equation_boson(x):
    return dos_prefactor(T0) * mp.polylog(2, mp.exp(x))
def N_equation_fermion(x):
    return -dos_prefactor(T0) * mp.polylog(3, -mp.exp(x)) - N0
def dN_equation_fermion(x):
    return -dos_prefactor(T0) * mp.polylog(2, -mp.exp(x))

## 3. Compute Initial Chemical Potentials & Ω₀

In [ ]:
# Polylogarithm orders for the harmonic oscillator
s_N = 3;  s_gbar_N = 2       # NOTE: code convention (PDF eq.48 gives 5/2)
s_Omega = 4;  s_gbar_Omega = 7/2
p_exp = 3;  nu_half = 3

# --- Bosons ---
start = time.time()
alpha_b = newton_raphson_1var(N_equation_boson, dN_equation_boson, -12, -11.5, dx=1e-5)
if alpha_b is None: raise RuntimeError("Root not found for boson mu")
mu0_b = alpha_b * kB * T0
Omega0_b = compute_initial_omega(+1, kB, T0, float(dos_prefactor(T0)), s_Omega, alpha_b)
E0_b = float(3 * N0 * kB * T0 * (mp.polylog(4, mp.exp(alpha_b)) / mp.polylog(3, mp.exp(alpha_b))))
print(f"Bosons:   alpha = {alpha_b:.10f},  mu = {mu0_b:.6e}")
print(f"          Omega0 = {Omega0_b:.6e},  E0 = {E0_b:.6e}  [{time.time()-start:.1f}s]")

# --- Fermions ---
start = time.time()
alpha_f = newton_raphson_1var(N_equation_fermion, dN_equation_fermion, -12, -11.5, dx=1e-5)
if alpha_f is None: raise RuntimeError("Root not found for fermion mu")
mu0_f = alpha_f * kB * T0
Omega0_f = compute_initial_omega(-1, kB, T0, float(dos_prefactor(T0)), s_Omega, alpha_f)
E0_f = float(3 * N0 * kB * T0 * (mp.polylog(4, -mp.exp(alpha_f)) / mp.polylog(3, -mp.exp(alpha_f))))
print(f"Fermions: alpha = {alpha_f:.10f},  mu = {mu0_f:.6e}")
print(f"          Omega0 = {Omega0_f:.6e},  E0 = {E0_f:.6e}  [{time.time()-start:.1f}s]")

## 4. Build Cut-off Schedule & Initialize Data

In [ ]:
Q0 = 5e-4; dQ = 1e-7; N_STEPS = 6000
Q_schedule = build_cutoff_schedule(Q0, dQ, N_STEPS)

results_b  = create_omega_result_dict()
results_f  = create_omega_result_dict()
results_mb = create_mb_result_dict()

initialize_omega_state(results_b, N0, T0, mu0_b, Omega0_b, E0_b)
initialize_omega_state(results_f, N0, T0, mu0_f, Omega0_f, E0_f)
initialize_mb_state(results_mb, N0, T0)

results_b['Q'] = Q_schedule; results_f['Q'] = Q_schedule
results_mb['Q'] = build_cutoff_schedule(Q0, dQ, N_STEPS + 1)

## 5. Fused Truncated Distribution Functions (N, Ω)

For the harmonic oscillator, the recurrences are:

$$\frac{N_1}{N_0} = \frac{\tilde{g}_{3}}{g_{3}}
  - \frac{2}{\sqrt{\pi}}\,\eta_c^{1/2}\,\frac{\bar{g}_{2}}{g_{3}}$$

$$\frac{\Omega_1}{\Omega_0} = \frac{\tilde{g}_{4}}{g_{4}}
  - \frac{2}{\sqrt{\pi}}\,\eta_c^{1/2}\,\frac{\bar{g}_{7/2}}{g_{4}}$$

In [ ]:
truncated_NOmega_boson_osc = make_truncated_NOmega(
    sign=+1, s_N=s_N, s_gbar_N=s_gbar_N,
    s_Omega=s_Omega, s_gbar_Omega=s_gbar_Omega, kB=kB)

truncated_NOmega_fermion_osc = make_truncated_NOmega(
    sign=-1, s_N=s_N, s_gbar_N=s_gbar_N,
    s_Omega=s_Omega, s_gbar_Omega=s_gbar_Omega, kB=kB)

## 6. Newton–Raphson (2-Variable) Solver for (N, Ω)

In [ ]:
nr_solver_boson_osc = make_nr_solver_omega(
    sign=+1, s_N=s_N, s_Omega=s_Omega, p_exp=p_exp,
    dos_prefactor_func=dos_prefactor, kB=kB)

nr_solver_fermion_osc = make_nr_solver_omega(
    sign=-1, s_N=s_N, s_Omega=s_Omega, p_exp=p_exp,
    dos_prefactor_func=dos_prefactor, kB=kB)

## 7. Run Evaporation Simulations

In [ ]:
eos_energy_boson = make_eos_energy(+1, s_N, s_Omega, nu_half, kB)
eos_energy_fermion = make_eos_energy(-1, s_N, s_Omega, nu_half, kB)

dT_nr = 1e-16; dmu_nr = 1e-18

print("Running boson evaporation (N, Omega)...")
start = time.time()
run_quantum_evaporation_omega(results_b, truncated_NOmega_boson_osc,
                              nr_solver_boson_osc, N0, N_STEPS, dT_nr, dmu_nr,
                              eos_energy_func=eos_energy_boson)
print(f"  Done in {time.time()-start:.1f}s")

print("Running fermion evaporation (N, Omega)...")
start = time.time()
run_quantum_evaporation_omega(results_f, truncated_NOmega_fermion_osc,
                              nr_solver_fermion_osc, N0, N_STEPS, dT_nr, dmu_nr,
                              eos_energy_func=eos_energy_fermion)
print(f"  Done in {time.time()-start:.1f}s")

print("Running Maxwell-Boltzmann evaporation...")
start = time.time()
run_mb_evaporation(results_mb, mb_particle_number, mb_temperature_oscillator, N0, N_STEPS + 1)
print(f"  Done in {time.time()-start:.1f}s")

## 8. Results

In [ ]:
fig = plot_combined_overview(results_b, results_f, results_mb, "Harmonic Oscillator (N, Omega)")
plt.show()

In [ ]:
fig = plot_individual_panels(results_b, results_f, results_mb, "Harmonic Oscillator (N, Omega)")
plt.show()